# DFN-Based Optimization of NFPP Sodium-Ion Cells within an Integrated Plant–Network Digital Twin Framework for Solar–BESS Microgrids

This notebook implements the complete research pipeline for the DFN-based optimization and the multi-feeder network state realization and anomaly detection framework.

In [ ]:
import os
import subprocess
import sys
from getpass import getpass

# Environment Setup
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('sodium-ion-ess'):
        get_ipython().system('git clone https://github.com/mhizterpaul/sodium-ion-ess.git')
        get_ipython().run_line_magic('cd', 'sodium-ion-ess')
    sys.path.append(os.getcwd())

# MP API Key configuration
if 'MP_API_KEY' not in os.environ:
    os.environ['MP_API_KEY'] = getpass("Enter Materials Project API Key: ")

!pip install pybamm numpy scipy matplotlib requests mp-api pymatgen pymoo mpi4py pint ufl
!add-apt-repository -y ppa:fenics-packages/fenics
!apt update
!apt install -y fenicsx
import pybamm
import numpy as np
import matplotlib.pyplot as plt
print("Environment initialized.")

## Stage 1: Cell Verification
Verify the base parameter set against literature-referenced material properties.

In [ ]:
from nfpp_sodium_ion.src.calibration.verify_parameters import verify

print("Stage 1: Running Cell Verification...")
verify()

## Stage 2: Cell Optimization
Hierarchical Material Discovery + Structural Sensitivity Optimization.

In [ ]:
from src.cell_optimization.parameter_opts import HierarchicalOptimizer

print("Stage 2: Running Hierarchical Material & Structural Optimization...")
optimizer = HierarchicalOptimizer()
optimized_res = optimizer.run()


## Stage 3: Stability Validation & Parameter Extraction
Performance evaluation and resistance profile generation for the digital twin.

In [ ]:
from src.cell_optimization.validate import OptimizationValidator

print("Stage 3: Running Stability Validation...")

# Map optimized design vector to parameter dict
design_specs = optimized_res.get("design_specs_representative", {})
deltas = optimized_res.get("combined_deltas_representative", {})

validator = OptimizationValidator(design_specs, deltas, engine=optimizer.engine)
results = validator.run_validation()

# Persist validation artifact for Stage 3.1 and Stage 4
import json
with open("final_validation.json", "w") as f:
    json.dump({"optimization": optimized_res, "validation": results}, f, indent=2)

print("\nStage 3.1: Running Parameter Extraction for Simscape...")
from src.simulation.envelope import StabilityValidator
stab_validator = StabilityValidator()
envelope_res = stab_validator.validate_optimized_design()
stab_validator.export_to_json(envelope_res)

print("\nPerformance Metrics Summary:")
if results:
    for k, v in results.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

## Stage 4: Multi-Feeder Digital Twin & Network State Realization
Assemble the multi-feeder shared source digital twin and execute network state realization for anomaly detection using phase dynamics.

In [ ]:
print("Stage 4.1: Assembling Multi-Feeder Plant Digital Twin...")
matlab_build_cmd = "matlab -nodisplay -nosplash -r \"addpath('src/power_plant'); build_physical_plant(struct()); quit;\""

try:
    subprocess.run(matlab_build_cmd, shell=True, check=True)
    print("Digital twin structure generated.")
except subprocess.CalledProcessError:
    print("MATLAB execution skipped (Environment check).")

print("\nStage 4.2: Running Multi-Feeder State Realization (Phase Dynamics)...")
matlab_sim_cmd = "matlab -nodisplay -nosplash -r \"addpath('src/simulation'); network_simulator(80, 20); quit;\""
try:
    subprocess.run(matlab_sim_cmd, shell=True, check=True)
    print("Network realization results exported to network_realization.json.")
except subprocess.CalledProcessError:
    print("MATLAB simulation skipped (Environment check). Components are ready in src/simulation/.")